<a href="https://colab.research.google.com/github/samtsa094/rps_model/blob/main/image_classification_rps_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import shutil
import cv2
import os

In [2]:
url = "https://www.kaggle.com/api/v1/datasets/download/sanikamal/rock-paper-scissors-dataset"
dataset = tf.keras.utils.get_file("rock_paper_scissors", url,
                                    extract=True, cache_dir='.',
                                    cache_subdir='')

474255368/474255368 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [3]:
dataset_dir = "/content/rock_paper_scissors/rock-paper-scissors/Rock-Paper-Scissors"
train_dir = os.path.join(dataset_dir, "train")
test_dir = os.path.join(dataset_dir, "test")
print(os.listdir(dataset_dir))
print(os.listdir(train_dir))
print(os.listdir(test_dir))

['train', 'validation', 'test']
['rock', 'paper', 'scissors']
['rock', 'paper', 'scissors']


In [4]:
train_dataset = tf.keras.utils.image_dataset_from_directory(train_dir, batch_size = 32, validation_split = 0.2, subset = "training", seed = 123, image_size = (150, 150))
validation_dataset = tf.keras.utils.image_dataset_from_directory(train_dir, batch_size = 32, validation_split = 0.2, subset = "validation", seed = 123, image_size = (150, 150))
test_dataset = tf.keras.utils.image_dataset_from_directory(test_dir, batch_size = 32, image_size = (150, 150))

Found 2520 files belonging to 3 classes.
Using 2016 files for training.
Found 2520 files belonging to 3 classes.
Using 504 files for validation.
Found 372 files belonging to 3 classes.


In [5]:
train_dataset = train_dataset.cache().prefetch(buffer_size = tf.data.AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size = tf.data.AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size = tf.data.AUTOTUNE)

In [6]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape = (150, 150, 3)),
    tf.keras.layers.Rescaling(1. / 255),
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),

    tf.keras.layers.Conv2D(32, (3, 3), activation = 'relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation = 'relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation = 'relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation = 'relu'),
    tf.keras.layers.Dense(3, activation = 'softmax')
])

In [7]:
model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])
model.fit(train_dataset, epochs = 10, validation_data = validation_dataset)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 91ms/step - accuracy: 0.5813 - loss: 1.0394 - val_accuracy: 0.9127 - val_loss: 0.2445
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.8492 - loss: 0.3701 - val_accuracy: 0.9206 - val_loss: 0.1716
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.9494 - loss: 0.1679 - val_accuracy: 0.9563 - val_loss: 0.1226
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.9697 - loss: 0.0886 - val_accuracy: 0.9881 - val_loss: 0.0432
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.9742 - loss: 0.0851 - val_accuracy: 0.9960 - val_loss: 0.0228
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.9782 - loss: 0.0741 - val_accuracy: 0.9802 - val_loss: 0.0526
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.9881 - loss: 0.0453 - val_accuracy: 0.9960 - val_loss: 0.0257
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.9831 - loss: 0.0487 - val_accuracy: 0.9980 - 

In [8]:
result = model.evaluate(test_dataset)
print(result)

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.8817 - loss: 0.2809
[0.2808546721935272, 0.8817204236984253]


In [9]:
model.save("rps_tf_model.keras")

In [10]:
model = tf.keras.models.load_model("rps_tf_model.keras")
print(model)

<Sequential name=sequential, built=True>


In [11]:
# import numpy as np
# import joblib
# import cv2
classes = ['Paper', 'Rock', 'Scissors']
model = tf.keras.models.load_model("rps_tf_model.keras")
print(model)
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    roi = frame[100: 400, 100: 400]
    img = cv2.resize(roi, (150, 150))
    img = np.expand_dims(img, axis = 0)
    prediction = model.predict(img, verbose = 0)
    class_idx = np.argmax(prediction)
    cv2.rectangle(frame, (100, 100), (400, 400), (255, 0, 0), 2)
    cv2.putText(frame, f"{classes[class_idx]}", (100, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.imshow('RPS Detection', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    cap.release()
    cv2.destroyAllWindows()

<Sequential name=sequential, built=True>


In [12]:
from IPython.display import Image
from IPython.display import clear_output
model = tf.keras.models.load_model("rps_tf_model.keras")
labels = ['Paper', 'Rock', 'Scissors']
try:
  while True:
    filename = take_photo()
    # Show the image which was just taken.
    display(Image(filename))
    image_file = cv2.imread(filename)
    image_file = cv2.resize(image_file, (150, 150))
    image = tf.expand_dims(image_file, 0)
    prediction = model.predict(image)
    print(labels[np.argmax(prediction)])
    clear_output()
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

name 'take_photo' is not defined


In [15]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      div.style.position = 'relative';
      div.style.width = 'max-content';

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      const boxSize = 200;
      const box = document.createElement('div');
      box.style.position = 'absolute';
      box.style.border = '4px solid red';
      box.style.width = boxSize + 'px';
      box.style.height = boxSize + 'px';
      box.style.top = '50%';
      box.style.left = '50%';
      box.style.transform = 'translate(-50%, -50%)';
      box.style.pointerEvents = 'none';
      div.appendChild(box);

      const label = document.createElement('div');
      label.innerText = 'CENTER GESTURE HERE - PRESS "D"';
      label.style.position = 'absolute';
      label.style.top = '-43px';
      label.style.left = '50%';
      label.style.transform = 'translateX(-50%)';
      label.style.color = 'red';
      label.style.fontWeight = 'bold';
      label.style.fontSize = '17px';
      label.style.whiteSpace = 'nowrap';
      box.appendChild(label);

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      try {
        const result = await new Promise((resolve, reject) => {
          const onKeyDown = (event) => {
            if (event.key.toLowerCase() === 'd') {
              window.removeEventListener('keydown', onKeyDown);

              const scaleX = video.videoWidth / video.offsetWidth;
              const scaleY = video.videoHeight / video.offsetHeight;

              const cropWidth = boxSize * scaleX;
              const cropHeight = boxSize * scaleY;
              const startX = (video.videoWidth - cropWidth) / 2;
              const startY = (video.videoHeight - cropHeight) / 2;

              const canvas = document.createElement('canvas');
              canvas.width = cropWidth;
              canvas.height = cropHeight;

              canvas.getContext('2d').drawImage(
                video,
                startX, startY, cropWidth, cropHeight,
                0, 0, cropWidth, cropHeight
              );

              resolve(canvas.toDataURL('image/jpeg', quality));
            }
          };
          window.addEventListener('keydown', onKeyDown);
        });
        return result;
      } finally {
        stream.getVideoTracks()[0].stop();
        div.remove();
      }
    }
    ''')
  display(js)
  try:
    data = eval_js('takePhoto({})'.format(quality))
  except KeyboardInterrupt:
    print('Camera closed.')
    return None

  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [17]:
def predict_gesture(img_path):
    if img_path is None:
        return None, None
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (150, 150))
    img_array = np.expand_dims(img_resized, axis=0)
    # Make prediction
    prediction = model.predict(img_array, verbose=0)
    class_idx = np.argmax(prediction)
    return labels[class_idx], prediction[0][class_idx]
try:
  print("Starting prediction loop. Press 'D' to capture or stop the cell to exit.")
  print()
  while True:
    filename = take_photo()
    if filename is None:
        print("Loop stopped by user.")
        break
    gesture, confidence = predict_gesture(filename)
    if gesture:
        print(f'Detected: {gesture} ({confidence*100:.2f}%)')
except Exception as err:
  print(f'Error: {err}')

Starting prediction loop. Press 'D' to capture or stop the cell to exit.



<IPython.core.display.Javascript object>

Detected: Rock (39.50%)


<IPython.core.display.Javascript object>

Detected: Paper (44.71%)


...

Detected: Rock (45.26%)


<IPython.core.display.Javascript object>

Detected: Paper (49.88%)


<IPython.core.display.Javascript object>

Camera closed.
Loop stopped by user.
